# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliHamza-lab/Ml_intership/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook turns the validated refresh queue into a practical content action playbook. It is written as a review aid for editors and analysts, not as an automated publishing system.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Ranked actions + reason codes

The ranked queue should be read as a shortlist for human review. In this repo, the top rows cluster around visible pages with declining demand and weak CTR or engagement signals, which is why the recommended actions are mostly refresh-and-review variants rather than full re-creation.

The reason codes in the exported queue map to simple editorial meanings:
- `declining_with_demand`: the page still has visible demand, but recent search behavior suggests it is losing momentum.
- `low_ctr_visible_page`: the page is visible and still receives impressions, but its CTR is weak relative to the rest of the queue.
- `low_engagement_visible_page`: the page attracts attention but is not converting that attention into engagement.
- `model_decline_risk`: the learned model assigned a high probability to decline.
- `visible_model_opportunity`: the page had enough visibility to make review worthwhile.
- `ctr_review_candidate`: the logical first step is a CTR-oriented refresh review.
- `engagement_review_candidate`: the logical first step is an engagement-oriented refresh review.


In [1]:
import pandas as pd
from pathlib import Path

queue_path = Path('../../outputs/refresh_queue.csv')
queue = pd.read_csv(queue_path)
print('queue rows', len(queue))
print('action mix')
print(queue['suggested_action'].value_counts().to_string())
print('\nTop 10 ranked rows')
print(queue[['final_rank','content_id','suggested_action','final_reason_codes','best_model_probability','impressions_90d','ctr','trend_direction']].head(10).to_string(index=False))


queue rows 30000
action mix
suggested_action
monitor                          13069
refresh                           8207
refresh_and_review_ctr            6655
refresh_and_review_engagement     1987
expand_and_refresh                  82

Top 10 ranked rows
 final_rank           content_id       suggested_action                                                                                                                                                   final_reason_codes  best_model_probability  impressions_90d  ctr trend_direction
          1 content_1f080331fa2b refresh_and_review_ctr declining_with_demand|low_ctr_visible_page|low_engagement_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate|engagement_review_candidate                0.786247            12834 0.05            down
          2 content_6aa43079fb0c refresh_and_review_ctr                                                         declining_with_demand|low_ctr_visible_page|model_decline_risk|

## 2. Intended use and limits

This playbook is intended for editorial triage. It should help a reviewer decide which existing content pages are worth inspecting first, and which type of refresh might be most appropriate. The evidence supports a ranking and a directional recommendation, not a guaranteed outcome.

Use it for:
- prioritizing a short review queue for content editors;
- surfacing pages that look visible enough to justify a refresh review;
- comparing a few candidate refresh interventions for the same page.

Do not use it for:
- fully automated publishing or direct content changes;
- claiming that a page will definitely decline or recover;
- replacing business or editorial judgment on brand fit, legal risk, or audience relevance.

The main limits are that the queue is based on one labeled snapshot and the model is validated on a grouped holdout rather than on a production rollout. It is therefore a decision-support aid, not a deployment guarantee.


In [2]:
import pandas as pd
from pathlib import Path

queue = pd.read_csv('../../outputs/refresh_queue.csv')
summary = {
    'rows_ranked': int(len(queue)),
    'high_confidence_rows': int((queue['confidence'] == 'high').sum()),
    'suggested_action_counts': queue['suggested_action'].value_counts().to_dict(),
    'reason_code_counts': pd.Series(' '.join(queue['final_reason_codes'].fillna('')).split()).value_counts().head(10).to_dict(),
}
print(summary)


{'rows_ranked': 30000, 'high_confidence_rows': 3576, 'suggested_action_counts': {'monitor': 13069, 'refresh': 8207, 'refresh_and_review_ctr': 6655, 'refresh_and_review_engagement': 1987, 'expand_and_refresh': 82}, 'reason_code_counts': {'general_refresh_review': 7701, 'page_one_decay_risk': 2364, 'declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate': 1962, 'declining_with_demand|model_decline_risk': 1560, 'declining_with_demand': 1337, 'declining_with_demand|model_decline_risk|visible_model_opportunity': 1152, 'general_refresh_review|model_decline_risk': 818, 'low_engagement_visible_page|engagement_review_candidate': 789, 'declining_with_demand|page_one_decay_risk|low_ctr_visible_page|visible_model_opportunity|ctr_review_candidate': 788, 'declining_with_demand|low_ctr_visible_page|visible_model_opportunity|ctr_review_candidate': 732}}


## 3. Human review + the no-go list

Before acting on a row, a human reviewer should check at least four things:
1. Is the page still relevant to the current audience and business goal?
2. Is the observed decline signal real, or is it a temporary fluctuation that the model has overfit to?
3. Does the page still have enough visibility to justify a refresh effort, or would the cost be disproportionate to the likely gain?
4. Does the suggested action align with the page’s current editorial purpose, not just the traffic trend?

The no-go list should be explicit:
- Do not auto-publish a rewrite or redirect from this queue alone.
- Do not automate outbound content changes for pages with recent brand, legal, or compliance sensitivity.
- Do not treat the queue as a ranking of page quality; it is a ranking of review priority.
- Do not use the queue to replace a content strategist’s judgment on topic fit or conversion strategy.


In [3]:
import pandas as pd
from pathlib import Path

queue = pd.read_csv('../../outputs/refresh_queue.csv')

def review_rules(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    frame['review_flag'] = (
        (frame['best_model_probability'] >= 0.75) |
        (frame['impressions_90d'] >= 5000) |
        (frame['ctr'] < 0.1)
    )
    return frame.loc[frame['review_flag']].head(20)

review_candidates = review_rules(queue)
print('review candidates')
print(review_candidates[['final_rank','content_id','suggested_action','best_model_probability','impressions_90d','ctr','days_since_last_update']].to_string(index=False))


review candidates
 final_rank           content_id              suggested_action  best_model_probability  impressions_90d  ctr  days_since_last_update
          1 content_1f080331fa2b        refresh_and_review_ctr                0.786247            12834 0.05                     104
          2 content_6aa43079fb0c        refresh_and_review_ctr                0.792117             8064 0.07                     104
          3 content_d6570c51c9bd        refresh_and_review_ctr                0.850354             2498 0.00                     104
          4 content_e04eb9549989        refresh_and_review_ctr                0.813830             3393 0.09                     104
          5 content_72e800a9c214        refresh_and_review_ctr                0.771037            13790 0.12                     104
          6 content_9b6df29f7889        refresh_and_review_ctr                0.847955             1622 0.12                     104
          7 content_b69288c5e701        refresh_and

## 4. Monitoring / retrain triggers

The playbook should be treated as dynamic. If the ranking becomes stale, the best signals will be changes in three areas:
- the share of top-ranked pages that actually look declining after the next review window;
- the drift in the feature distribution, especially visibility and CTR-related features;
- the gap between the queue’s early recommendations and what editors actually choose to refresh.

Practical retrain triggers:
- when the top 50 queue rows show a sharp drop in observed decline rate versus the prior month;
- when the label rate shifts materially because the underlying traffic mix changed;
- when the queue starts to over-index on one content type or one client and the ranking becomes less balanced.

A small monitoring dashboard is enough at first: count of rows by action, share of high-confidence rows, top reason-code mix, and observed decline rate in the next review window.


In [4]:
import pandas as pd
from pathlib import Path

queue = pd.read_csv('../../outputs/refresh_queue.csv')
print('Top action counts')
print(queue['suggested_action'].value_counts().to_string())
print('\nTop reason codes')
reason_series = queue['final_reason_codes'].fillna('').str.split('|').explode()
print(reason_series.value_counts().head(10).to_string())


Top action counts
suggested_action
monitor                          13069
refresh                           8207
refresh_and_review_ctr            6655
refresh_and_review_engagement     1987
expand_and_refresh                  82

Top reason codes
final_reason_codes
declining_with_demand          13152
visible_model_opportunity      10653
low_ctr_visible_page            9759
ctr_review_candidate            9759
general_refresh_review          9117
model_decline_risk              8459
page_one_decay_risk             7076
low_engagement_visible_page     6508
engagement_review_candidate     6508
thin_visible_page                 82


## 5. Exports for the paper

The paper will build from a small set of files that are easy to re-run and easy to audit. The notebook exports the ranked queue and a compact set of summary outputs under the project’s outputs folder so the paper can reference the same artifacts without relying on a hidden notebook state.

The queue CSV is the core export. It is regenerated from the existing pipeline and should be treated as a review artifact rather than a production system output.


In [7]:
from pathlib import Path
import pandas as pd

queue_path = Path('../../outputs/refresh_queue.csv')
output_dir = Path('../../work/outputs')
output_dir.mkdir(parents=True, exist_ok=True)

queue = pd.read_csv(queue_path)
# Keep the paper-facing exports under work/outputs.
queue.to_csv(output_dir / 'refresh_queue_for_paper.csv', index=False)

summary = {
    'rows_exported': int(len(queue)),
    'top_ranked_action': queue['suggested_action'].value_counts().idxmax(),
    'high_confidence_rows': int((queue['confidence'] == 'high').sum()),
    'queue_head': queue[['final_rank','content_id','suggested_action','final_reason_codes','best_model_probability']].head(10).to_dict(orient='records'),
}
Path(output_dir / 'action_playbook_summary.json').write_text(str(summary).replace("'", '"'))
print('Wrote', output_dir / 'refresh_queue_for_paper.csv')
print('Wrote', output_dir / 'action_playbook_summary.json')


Wrote ..\..\work\outputs\refresh_queue_for_paper.csv
Wrote ..\..\work\outputs\action_playbook_summary.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Demo outline for a 5-minute showcase

**Question:** Which existing FlyRank content pages should a reviewer inspect first when search visibility and engagement look weak?

**Method:** Use the anonymized starter dataset, build a transparent baseline, compare it with a learned ranker on a grouped client-holdout split, and translate the ranked output into editorial review actions.

**One chart:** A simple bar chart of the action mix or the top reason-code mix shows that the queue is dominated by refresh-and-review candidates rather than full re-creation.

**One honest result:** The model produced directional ranking signals for visible pages with weak CTR or engagement, but the ranking should be treated as decision-support rather than a guarantee of future decline.

**One recommendation:** Start with the top-ranked pages that combine visible demand with weak CTR or engagement, then ask a human reviewer to confirm relevance and editorial fit before acting.

## Shareable cuts

**Short social post:** I built a public-safe content refresh playbook that ranks existing pages for editorial review using an anonymized dataset and a grouped validation design. The method compares a transparent baseline with a learned ranker, then turns the output into reasoned refresh actions rather than automated publishing decisions. The result is a practical review queue that helps content teams focus on the pages most worth inspecting first.

**Employer-facing summary:** I built a decision-support playbook for FlyRank content refresh triage using the anonymized starter dataset and a client-held-out validation design. The work compared a transparent rule baseline with a learned ranker, then translated the ranked output into human-review actions with reason codes and limits. The result was a directional queue for prioritizing refresh review, not an automated publishing system.
